## Phase 1: Environment Setup & Data Preparation
In this phase, we prepare our workspace and consolidate our datasets. The raw sales data is divided into separate monthly files within a designated directory. 

**Objectives for this phase:**
1. Import essential libraries (`pandas`, `os`).
2. Iterate through the target directory to locate all monthly `.csv` files.
3. Concatenate the individual datasets into one comprehensive dataframe.
4. Export the master dataset for the upcoming analytical phases.

In [26]:
# Import essential libraries for data manipulation and system path management
import pandas as pd
import os

In [27]:
# Define the path to the directory containing the monthly sales datasets
folder_path = "salesmonths"

# Extract and display the list of files in the directory to verify their presence
files = os.listdir(folder_path)
files

['sales_2023_01.csv',
 'sales_2023_02.csv',
 'sales_2023_03.csv',
 'sales_2023_04.csv',
 'sales_2023_05.csv',
 'sales_2023_06.csv',
 'sales_2023_07.csv',
 'sales_2023_08.csv',
 'sales_2023_09.csv',
 'sales_2023_10.csv',
 'sales_2023_11.csv',
 'sales_2023_12.csv',
 'sales_2024_01.csv',
 'sales_2024_02.csv',
 'sales_2024_03.csv',
 'sales_2024_04.csv',
 'sales_2024_05.csv',
 'sales_2024_06.csv',
 'sales_2024_07.csv',
 'sales_2024_08.csv',
 'sales_2024_09.csv',
 'sales_2024_10.csv',
 'sales_2024_11.csv',
 'sales_2024_12.csv',
 'sales_2025_01.csv',
 'sales_2025_02.csv',
 'sales_2025_03.csv',
 'sales_2025_04.csv',
 'sales_2025_05.csv',
 'sales_2025_06.csv',
 'sales_2025_07.csv',
 'sales_2025_08.csv',
 'sales_2025_09.csv',
 'sales_2025_10.csv',
 'sales_2025_11.csv',
 'sales_2025_12.csv']

In [28]:
# Define the directory containing the monthly datasets
folder_path = "salesmonths"

# Initialize an empty list to collect individual monthly dataframes
dfs = []

# Iterate through all files within the specified directory
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_path, file))
        dfs.append(df)
        
# Concatenate all monthly dataframes into a single, unified dataframe
final_df = pd.concat(dfs, ignore_index=True)

In [29]:
# Export the consolidated dataframe to a new CSV file for future analysis
final_df.to_csv("all_sales.csv", index=False)


In [30]:
final_df.shape

(1080000, 13)

## Phase 2: Data Preparation & Feature Engineering
Before diving into exploratory data analysis, we must sanitize and enrich our dataset. This section covers the essential data preparation steps:

1. **Data Profiling:** Checking data structures and identifying duplicate records.
2. **Date Parsing:** Standardizing the `sale_date` column and extracting temporal features.
3. **Text Cleaning:** Formatting categorical variables (e.g., `unit_type`, `agent`) for consistency.
4. **Calculated Fields:** Creating new business-relevant metrics (`payment_years`, `down_payment_pct`).

In [31]:
# Load the consolidated master dataset into a pandas dataframe for analysis
df = pd.read_csv("all_sales.csv")

# Preview the dataframe to inspect its structure, columns, and initial rows
df

,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,channel,agent,status
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3BR,6691492,100000,6591492,8Y,938086.13,Direct,Mona Adel,Pending
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,Broker,Sara Khaled,Canceled
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,Direct,Mona Adel,Pending
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2BR,2363226,100000,2263226,Cash,554752.04,Online,Omar Tarek,Completed
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3BR,5212700,0,5212700,5Y,1536354.55,Direct,Youssef Samy,Pending
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079995,S2025120029995,2025-12-21,CUST_6350,PRJ_10,Studio,8435479,50000,8385479,8Y,2133698.61,Online,Omar Tarek,Completed
1079996,S2025120029996,2025-12-25,CUST_9482,PRJ_7,3BR,1473538,150000,1323538,5Y,376313.27,Online,Heba Mostafa,Canceled
1079997,S2025120029997,2025-12-12,CUST_12868,PRJ_9,Villa,6203054,100000,6103054,Cash,1012280.01,Direct,Heba Mostafa,Completed
1079998,S2025120029998,2025-12-22,CUST_5354,PRJ_24,3BR,2953056,0,2953056,Cash,737956.47,Broker,Ahmed Hassan,Completed


In [32]:
# Display a concise summary of the dataframe to inspect data types and identify any missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1080000 entries, 0 to 1079999
Data columns (total 13 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   sale_id       1080000 non-null  object 
 1   sale_date     1080000 non-null  object 
 2   customer_id   1080000 non-null  object 
 3   project_id    1080000 non-null  object 
 4   unit_type     1080000 non-null  object 
 5   unit_price    1080000 non-null  int64  
 6   discount      1080000 non-null  int64  
 7   final_price   1080000 non-null  int64  
 8   payment_plan  1080000 non-null  object 
 9   down_payment  1080000 non-null  float64
 10  channel       1080000 non-null  object 
 11  agent         1080000 non-null  object 
 12  status        1080000 non-null  object 
dtypes: float64(1), int64(3), object(9)
memory usage: 107.1+ MB


In [33]:
# Count the total number of duplicate 'sale_id' values
# This validation step ensures each transaction is unique and prevents double-counting in our analysis
df['sale_id'].duplicated().sum()

np.int64(0)

In [34]:
# Convert the 'sale_date' column to proper datetime objects
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Extract specific temporal features (year and numeric month) 
df['year'] = df['sale_date'].dt.year
df['month'] = df['sale_date'].dt.month

# Extract the categorical month name to enhance readability and label clarity in future visualizations
df['month_name'] = df['sale_date'].dt.month_name()
df.head()

,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,channel,agent,status,year,month,month_name
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3BR,6691492,100000,6591492,8Y,938086.13,Direct,Mona Adel,Pending,2023,1,January
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,Broker,Sara Khaled,Canceled,2023,1,January
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,Direct,Mona Adel,Pending,2023,1,January
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2BR,2363226,100000,2263226,Cash,554752.04,Online,Omar Tarek,Completed,2023,1,January
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3BR,5212700,0,5212700,5Y,1536354.55,Direct,Youssef Samy,Pending,2023,1,January


In [35]:
# Define categorical text columns that require standardization
text_cols = ['unit_type', 'payment_plan', 'channel', 'agent', 'status']

# Loop through each column to apply consistent text formatting
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

In [36]:
# Inspect the unique values within the 'payment_plan' column
df['payment_plan'].unique()

array(['8Y', 'Cash', '5Y'], dtype=object)

In [37]:
# Transform the categorical 'payment_plan' into a quantitative 'payment_years' feature
df['payment_years'] = (
    df['payment_plan']
      .replace({'Cash': '0'})
      .str.replace('Y', '', regex=False)
      .astype(int)
)
df

,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,channel,agent,status,year,month,month_name,payment_years
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3Br,6691492,100000,6591492,8Y,938086.13,Direct,Mona Adel,Pending,2023,1,January,8
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,Broker,Sara Khaled,Canceled,2023,1,January,0
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,Direct,Mona Adel,Pending,2023,1,January,8
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2Br,2363226,100000,2263226,Cash,554752.04,Online,Omar Tarek,Completed,2023,1,January,0
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3Br,5212700,0,5212700,5Y,1536354.55,Direct,Youssef Samy,Pending,2023,1,January,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079995,S2025120029995,2025-12-21,CUST_6350,PRJ_10,Studio,8435479,50000,8385479,8Y,2133698.61,Online,Omar Tarek,Completed,2025,12,December,8
1079996,S2025120029996,2025-12-25,CUST_9482,PRJ_7,3Br,1473538,150000,1323538,5Y,376313.27,Online,Heba Mostafa,Canceled,2025,12,December,5
1079997,S2025120029997,2025-12-12,CUST_12868,PRJ_9,Villa,6203054,100000,6103054,Cash,1012280.01,Direct,Heba Mostafa,Completed,2025,12,December,0
1079998,S2025120029998,2025-12-22,CUST_5354,PRJ_24,3Br,2953056,0,2953056,Cash,737956.47,Broker,Ahmed Hassan,Completed,2025,12,December,0


In [38]:
# Calculate the down payment percentage relative to the total unit price
df['down_payment_pct'] = (df['down_payment'] / df['unit_price']) * 100
df

,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,channel,agent,status,year,month,month_name,payment_years,down_payment_pct
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3Br,6691492,100000,6591492,8Y,938086.13,Direct,Mona Adel,Pending,2023,1,January,8,14.019088
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,Broker,Sara Khaled,Canceled,2023,1,January,0,11.013890
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,Direct,Mona Adel,Pending,2023,1,January,8,14.467552
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2Br,2363226,100000,2263226,Cash,554752.04,Online,Omar Tarek,Completed,2023,1,January,0,23.474354
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3Br,5212700,0,5212700,5Y,1536354.55,Direct,Youssef Samy,Pending,2023,1,January,5,29.473297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079995,S2025120029995,2025-12-21,CUST_6350,PRJ_10,Studio,8435479,50000,8385479,8Y,2133698.61,Online,Omar Tarek,Completed,2025,12,December,8,25.294338
1079996,S2025120029996,2025-12-25,CUST_9482,PRJ_7,3Br,1473538,150000,1323538,5Y,376313.27,Online,Heba Mostafa,Canceled,2025,12,December,5,25.538077
1079997,S2025120029997,2025-12-12,CUST_12868,PRJ_9,Villa,6203054,100000,6103054,Cash,1012280.01,Direct,Heba Mostafa,Completed,2025,12,December,0,16.319058
1079998,S2025120029998,2025-12-22,CUST_5354,PRJ_24,3Br,2953056,0,2953056,Cash,737956.47,Broker,Ahmed Hassan,Completed,2025,12,December,0,24.989586


In [39]:
# Export the fully cleaned and engineered dataset to a new CSV file
df.to_csv("sales_cleaned.csv", index=False)

## Phase 3: Secondary Dataset Ingestion & Dimension Cleaning (Projects Data)

This phase focuses on ensuring structural integrity, removing logical data anomalies, and performing feature engineering to segment projects by market value and lifecycle status.

**Key Objectives for this Phase:**
* **Dimension Profiling:** Ingesting the Excel sheet and validating that the primary key (`project_id`) is strictly unique to support accurate data relationships.
* **Text Uniformity:** Standardizing text fields (Location, Developer, Project Type) to ensure consistent grouping and seamless filtering.
* **Logical Validation & Filtering:** Eliminating operational anomalies, such as negative unit inventories or inverted pricing boundaries.
* **Advanced Feature Engineering:**
  * Deriving descriptive metrics: `price_range` and `avg_price`.
  * Implementing **Data Segmentation** (`price_category`) to establish clean pricing tiers (Low, Medium, High) for executive reporting.
  * Standardizing project status into a unified **Business Taxonomy** (`status`) to streamline dashboard visualization.

In [40]:
# 1) Load the supplementary projects dataset from an Excel file
# Explicitly targeting the "projects.csv" sheet ensures we extract the exact data required for this analysis phase
df = pd.read_excel(
    "projects.xlsx",
    sheet_name="projects.csv"
)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   project_id     40 non-null     object
 1   project_name   40 non-null     object
 2   location       40 non-null     object
 3   developer      40 non-null     object
 4   project_type   40 non-null     object
 5   start_price    40 non-null     int64 
 6   max_price      40 non-null     int64 
 7   units_count    40 non-null     int64 
 8   delivery_year  40 non-null     int64 
 9   status         40 non-null     object
dtypes: int64(4), object(6)
memory usage: 3.3+ KB


In [42]:
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready
3,PRJ_4,Project 4,October,SODIC,Residential,2517796,5379090,289,2028,Ready
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched


In [43]:
# 2) Count the total number of duplicate 'project_id' values
df['project_id'].duplicated().sum()

np.int64(0)

In [44]:
df.select_dtypes(include=object)

,project_id,project_name,location,developer,project_type,status
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,Ready
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,Launched
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,Ready
3,PRJ_4,Project 4,October,SODIC,Residential,Ready
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,Launched
5,PRJ_6,Project 6,Maadi,Ora Developers,Mixed Use,Ready
6,PRJ_7,Project 7,North Coast,Mountain View,Mixed Use,Launched
7,PRJ_8,Project 8,Zagazig,Ora Developers,Commercial,Ready
8,PRJ_9,Project 9,North Coast,SODIC,Administrative,Launched
9,PRJ_10,Project 10,New Cairo,Hassan Allam,Mixed Use,Ready


In [45]:
# 3) Clean all text columns
# Define categorical text columns within the projects dataset that require standardization
text_cols = [
    'project_name', 'location',
    'developer', 'project_type', 'status']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched


In [46]:
# 4) Calculate the 'price_range' by finding the difference between the maximum and starting prices

df['price_range'] = df['max_price'] - df['start_price']
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready,4204102
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched,2023927
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready,6664778
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready,2861294
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched,7259265


In [47]:
#5) Calculate the 'avg_price' to establish a standardized valuation baseline for each project

df['avg_price'] = (df['max_price'] + df['start_price']) / 2
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range,avg_price
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready,4204102,4292075.0
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched,2023927,2759780.5
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready,6664778,5371337.0
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready,2861294,3948443.0
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched,7259265,4934832.5


In [48]:
# 6) Filter out anomalous records where the starting price exceeds the maximum price

df = df[df['max_price'] >= df['start_price']]
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range,avg_price
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready,4204102,4292075.0
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched,2023927,2759780.5
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready,6664778,5371337.0
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready,2861294,3948443.0
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched,7259265,4934832.5


In [49]:
# 7) Filter out projects with zero or negative unit counts
df = df[df['units_count'] > 0]
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range,avg_price
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready,4204102,4292075.0
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched,2023927,2759780.5
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready,6664778,5371337.0
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready,2861294,3948443.0
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched,7259265,4934832.5


In [50]:
# 8) Segment projects into distinct pricing tiers based on their average price

df['price_category'] = pd.cut(
    df['avg_price'],
    bins=[0, 2000000, 5000000, 10000000],
    labels=['Low', 'Medium', 'High']
)
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range,avg_price,price_category
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Ready,4204102,4292075.0,Medium
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Launched,2023927,2759780.5,Medium
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Ready,6664778,5371337.0,High
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Ready,2861294,3948443.0,Medium
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Launched,7259265,4934832.5,Medium


In [51]:
df['status'].unique()

array(['Ready', 'Launched', 'Under Construction'], dtype=object)

In [52]:
# 9) Status Cleaning & Standardization
# Consolidate project status categories into a standardized business taxonomy

df['status'] = df['status'].replace({
    'Launched': 'Under Development',
    'Ready': 'Delivered'
})

# Preview the dataframe to confirm the status values have been successfully unified
df.head()

,project_id,project_name,location,developer,project_type,start_price,max_price,units_count,delivery_year,status,price_range,avg_price,price_category
0,PRJ_1,Project 1,Nasr City,Palm Hills,Residential,2190024,6394126,195,2025,Delivered,4204102,4292075.0,Medium
1,PRJ_2,Project 2,Mansoura,Tatweer Misr,Mixed Use,1747817,3771744,586,2027,Under Development,2023927,2759780.5,Medium
2,PRJ_3,Project 3,Mansoura,Tatweer Misr,Residential,2038948,8703726,879,2027,Delivered,6664778,5371337.0,High
3,PRJ_4,Project 4,October,Sodic,Residential,2517796,5379090,289,2028,Delivered,2861294,3948443.0,Medium
4,PRJ_5,Project 5,Sohag,Emaar,Mixed Use,1305200,8564465,176,2026,Under Development,7259265,4934832.5,Medium


In [53]:
df['status'].unique()

array(['Delivered', 'Under Development', 'Under Construction'],
      dtype=object)

In [54]:
# 10) Export the fully cleaned and engineered projects dimension table to a new CSV file
df.to_csv("projects_cleaned.csv", index=False)

## Phase 4: Customer Dimension Cleaning & Comprehensive Standardization

This phase focuses on ingesting and standardizing the customer dimension table. The objective is to ensure structural integrity, clean Personally Identifiable Information (PII), translate categorical variables, and perform advanced geospatial and temporal feature engineering to support seamless dashboard filtering.

**Key Objectives for this Phase:**
* **Dimension Profiling:** Ingesting the dataset and validating the uniqueness of the primary key (`customer_id`) to maintain robust dimension-to-fact relationships.
* **PII & Text Uniformity:** 
  * Applying defensive string formatting (strip, title, lower) to text columns and emails.
  * Utilizing Regular Expressions (Regex `\D`) to extract and standardize numeric phone values.
* **Categorical Translation & Taxonomy:** Standardizing bilingual categorical fields (`customer_type`, `lead_source`) by translating Arabic entries into a unified English taxonomy, preventing encoding issues in downstream BI tools.
* **Temporal Feature Engineering:** Converting raw `created_at` strings into precise datetime objects, and extracting distinct temporal dimensions (`created_year`, `created_month`) to enable time-series analysis and storytelling.
* **Advanced Geospatial Normalization:** Implementing hierarchical tuple mapping to transform unstructured, messy address strings into clean, standardized `city` and `governorate` columns, effectively reducing noise for high-level spatial analysis.

In [55]:
# 1) Load the third foundational dataset: The Customers Dimension Table

df = pd.read_excel(
    "customers_realistic_address.xlsx",
    sheet_name="customers_realistic_address.csv"
)
df.head()

,customer_id,full_name,phone,email,address,customer_type,lead_source,budget_range,created_at,sales_agent,segment
0,CUST_1,Customer 1,1077415596,customer1@mail.com,"الجيزة, SheikhZayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,2025-05-27,Mohamed,Normal
1,CUST_2,Customer 2,1020857153,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",شركه,Event,5M+,2024-05-02,Ahmed,Normal
2,CUST_3,Customer 3,1087052530,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,واتساب,2-3M,2023-06-30,Ahmed,New
3,CUST_4,Customer 4,1070589979,customer4@mail.com,"Assiut, Assiut East, Helaly, building 80",فرد,واتساب,1-2M,2025-06-22,Ali,VIP
4,CUST_5,Customer 5,1081852995,customer5@mail.com,"اسيوط, AssiutEast, El Geish, B60",Company,Referral,3-5M,2025-07-17,Mohamed,Normal


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    30000 non-null  object        
 1   full_name      30000 non-null  object        
 2   phone          30000 non-null  int64         
 3   email          30000 non-null  object        
 4   address        30000 non-null  object        
 5   customer_type  30000 non-null  object        
 6   lead_source    30000 non-null  object        
 7   budget_range   30000 non-null  object        
 8   created_at     30000 non-null  datetime64[ns]
 9   sales_agent    30000 non-null  object        
 10  segment        30000 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 2.5+ MB


In [57]:
# 2) Count the total number of duplicate 'customer_id' values
df['customer_id'].duplicated().sum()

np.int64(0)

In [58]:
df.select_dtypes(include=object)

,customer_id,full_name,email,address,customer_type,lead_source,budget_range,sales_agent,segment
0,CUST_1,Customer 1,customer1@mail.com,"الجيزة, SheikhZayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,Mohamed,Normal
1,CUST_2,Customer 2,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",شركه,Event,5M+,Ahmed,Normal
2,CUST_3,Customer 3,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,واتساب,2-3M,Ahmed,New
3,CUST_4,Customer 4,customer4@mail.com,"Assiut, Assiut East, Helaly, building 80",فرد,واتساب,1-2M,Ali,VIP
4,CUST_5,Customer 5,customer5@mail.com,"اسيوط, AssiutEast, El Geish, B60",Company,Referral,3-5M,Mohamed,Normal
...,...,...,...,...,...,...,...,...,...
29995,CUST_29996,Customer 29996,customer29996@mail.com,"Dakahlia, Mansoura, Gomhoria, building 65",Company,Event,1-2M,Sara,Normal
29996,CUST_29997,Customer 29997,customer29997@mail.com,"الجيزه, Dokki, Mosadak, B10",Individual,Facebook,3-5M,Ali,Normal
29997,CUST_29998,Customer 29998,customer29998@mail.com,"Suez, Arbeen, El Geish, B62",Company,Call,1-2M,Ahmed,Normal
29998,CUST_29999,Customer 29999,customer29999@mail.com,"الجيزه, Mohandseen, Gameat El Dowal, building 59",فرد,Facebook,5M+,Ahmed,VIP


In [59]:
# 3) Clean text columns
# Define categorical text columns within the projects dataset that require standardization
text_cols = [
    'full_name', 'address',
    'sales_agent', 'segment']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

df.head()

,customer_id,full_name,phone,email,address,customer_type,lead_source,budget_range,created_at,sales_agent,segment
0,CUST_1,Customer 1,1077415596,customer1@mail.com,"الجيزة, Sheikhzayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,2025-05-27,Mohamed,Normal
1,CUST_2,Customer 2,1020857153,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",شركه,Event,5M+,2024-05-02,Ahmed,Normal
2,CUST_3,Customer 3,1087052530,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,واتساب,2-3M,2023-06-30,Ahmed,New
3,CUST_4,Customer 4,1070589979,customer4@mail.com,"Assiut, Assiut East, Helaly, Building 80",فرد,واتساب,1-2M,2025-06-22,Ali,Vip
4,CUST_5,Customer 5,1081852995,customer5@mail.com,"اسيوط, Assiuteast, El Geish, B60",Company,Referral,3-5M,2025-07-17,Mohamed,Normal


In [60]:
# 4) Email Cleaning
df['email'] = df['email'].astype(str).str.strip().str.lower()

In [61]:
# 5) Clean the 'phone' column using Regular Expressions (Regex)
# The '\D' pattern matches any non-digit character (e.g., spaces, dashes, parentheses) and removes it

df['phone'] = df['phone'].astype(str).str.replace(r'\D', '', regex=True)

In [62]:
df['customer_type'].unique()

array(['Individual', 'شركه', 'فرد', 'Company'], dtype=object)

In [63]:
# 6) Standardize the 'customer_type' column by translating Arabic values to English

df['customer_type'] = df['customer_type'].replace({
    'فرد': 'Individual',
    'شركه': 'Company'
})

In [64]:
# 7) Apply string formatting to the 'customer_type' column as a defensive measure
df['customer_type'] = df['customer_type'].astype(str).str.strip().str.title()
df.head()

,customer_id,full_name,phone,email,address,customer_type,lead_source,budget_range,created_at,sales_agent,segment
0,CUST_1,Customer 1,1077415596,customer1@mail.com,"الجيزة, Sheikhzayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,2025-05-27,Mohamed,Normal
1,CUST_2,Customer 2,1020857153,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",Company,Event,5M+,2024-05-02,Ahmed,Normal
2,CUST_3,Customer 3,1087052530,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,واتساب,2-3M,2023-06-30,Ahmed,New
3,CUST_4,Customer 4,1070589979,customer4@mail.com,"Assiut, Assiut East, Helaly, Building 80",Individual,واتساب,1-2M,2025-06-22,Ali,Vip
4,CUST_5,Customer 5,1081852995,customer5@mail.com,"اسيوط, Assiuteast, El Geish, B60",Company,Referral,3-5M,2025-07-17,Mohamed,Normal


In [65]:
df['lead_source'].unique()

array(['Referral', 'Event', 'واتساب', 'Facebook', 'Call'], dtype=object)

In [66]:
# 8) Standardize the 'lead_source' column by translating Arabic entries to English
df['lead_source'] = df['lead_source'].replace({
    'واتساب': 'WhatsApp'
})
df.head()

,customer_id,full_name,phone,email,address,customer_type,lead_source,budget_range,created_at,sales_agent,segment
0,CUST_1,Customer 1,1077415596,customer1@mail.com,"الجيزة, Sheikhzayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,2025-05-27,Mohamed,Normal
1,CUST_2,Customer 2,1020857153,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",Company,Event,5M+,2024-05-02,Ahmed,Normal
2,CUST_3,Customer 3,1087052530,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,WhatsApp,2-3M,2023-06-30,Ahmed,New
3,CUST_4,Customer 4,1070589979,customer4@mail.com,"Assiut, Assiut East, Helaly, Building 80",Individual,WhatsApp,1-2M,2025-06-22,Ali,Vip
4,CUST_5,Customer 5,1081852995,customer5@mail.com,"اسيوط, Assiuteast, El Geish, B60",Company,Referral,3-5M,2025-07-17,Mohamed,Normal


In [67]:
# 9) Apply string formatting to the 'lead_source' column as a defensive measure
df['lead_source'] = df['lead_source'].astype(str).str.strip().str.title()
df.head()

,customer_id,full_name,phone,email,address,customer_type,lead_source,budget_range,created_at,sales_agent,segment
0,CUST_1,Customer 1,1077415596,customer1@mail.com,"الجيزة, Sheikhzayed, Youth Housing, عمارة 12",Individual,Referral,3-5M,2025-05-27,Mohamed,Normal
1,CUST_2,Customer 2,1020857153,customer2@mail.com,"الدقهلية, Mansoura, Portsaid, عمارة 64",Company,Event,5M+,2024-05-02,Ahmed,Normal
2,CUST_3,Customer 3,1087052530,customer3@mail.com,"Assiut, Assiut East, El Geish, عمارة 23",Individual,Whatsapp,2-3M,2023-06-30,Ahmed,New
3,CUST_4,Customer 4,1070589979,customer4@mail.com,"Assiut, Assiut East, Helaly, Building 80",Individual,Whatsapp,1-2M,2025-06-22,Ali,Vip
4,CUST_5,Customer 5,1081852995,customer5@mail.com,"اسيوط, Assiuteast, El Geish, B60",Company,Referral,3-5M,2025-07-17,Mohamed,Normal


In [68]:
# 10) Budget Range Cleaning
df['budget_range'] = df['budget_range'].astype(str).str.strip()

In [69]:
# 11) Temporal Standardization and Feature Extraction
# Convert the 'created_at' column to a proper datetime object 
df['created_at'] = pd.to_datetime(df['created_at'])

# Extract 'created_year' and 'created_month' to enable 
df['created_year'] = df['created_at'].dt.year
df['created_month'] = df['created_at'].dt.month

In [70]:
# .tolist() forces Python to print every single unique value vertically
df['address'].unique().tolist()

['الجيزة, Sheikhzayed, Youth Housing, عمارة 12',
 'الدقهلية, Mansoura, Portsaid, عمارة 64',
 'Assiut, Assiut East, El Geish, عمارة 23',
 'Assiut, Assiut East, Helaly, Building 80',
 'اسيوط, Assiuteast, El Geish, B60',
 'الجيزة, Dokki, Tahrir, عمارة 6',
 'السويس, Arbeen, El Geish, عمارة 24',
 'Suez, Arbeen, Elgeish, Building 53',
 'Alex, Miami, Khaledibnelwalid, Building 20',
 'السويس, Arbeen, Elgeish, Building 97',
 'Suez, Arbeen, Elgeish, عمارة 44',
 'Alex, Miami, Khaled Ibn El Walid, B90',
 'Alex, Smouha, Victoremanuel, عمارة 98',
 'الاسكندريه, Smouha, Victoremanuel, Building 98',
 'Dakahlia, Mansoura, Portsaid, B69',
 'Assiut, Assiuteast, Helaly, Building 93',
 'Dakahlia, Mansoura, Portsaid, Building 20',
 'أسيوط, Assiut East, Helaly, عمارة 98',
 'Giza, Dokki, Tahrir, عمارة 74',
 'Cairo, Maadi, Degla, B51',
 'الإسكندرية, Lauran, Abdelsalamaref, Building 99',
 'الإسكندرية, Smouha, Fawzy Moaz, B27',
 'القاهره, Nasr City, Makram Ebeid, Building 13',
 'الدقهلية, Mansoura, Portsaid, B77'

In [71]:

# Address Mapping Dictionary (Data Standardization)
# =======================================================
# Purpose: Converts messy Arabic/English addresses into clean English categories.
# Format:  'raw_input_text': ('City', 'Governorate')
# Benefit: Ensures accurate grouping and clean slicers for BI dashboards.

address_map = {
    # ===================== CAIRO – New Cairo =====================
    'new cairo': ('New Cairo', 'Cairo'),
    'newcairo': ('New Cairo', 'Cairo'),
    'القاهرة الجديدة': ('New Cairo', 'Cairo'),
    'التجمع الخامس': ('New Cairo', 'Cairo'),
    'fifth settlement': ('New Cairo', 'Cairo'),
    '90th street': ('New Cairo', 'Cairo'),
    '90th st': ('New Cairo', 'Cairo'),
    '90thstreet': ('New Cairo', 'Cairo'),
    'north 90': ('New Cairo', 'Cairo'),
    'south 90': ('New Cairo', 'Cairo'),
    'south ninety': ('New Cairo', 'Cairo'),
    '90 street': ('New Cairo', 'Cairo'),

    # ===================== CAIRO – Nasr City =====================
    'nasr city': ('Nasr City', 'Cairo'),
    'nasrcity': ('Nasr City', 'Cairo'),
    'مدينة نصر': ('Nasr City', 'Cairo'),
    'عباس العقاد': ('Nasr City', 'Cairo'),
    'abbas el akkad': ('Nasr City', 'Cairo'),
    'abbaselakkad': ('Nasr City', 'Cairo'),
    'makram ebeid': ('Nasr City', 'Cairo'),
    'makramebeid': ('Nasr City', 'Cairo'),
    'mostafa elnahas': ('Nasr City', 'Cairo'),

    # ===================== CAIRO – Maadi =====================
    'maadi': ('Maadi', 'Cairo'),
    'المعادي': ('Maadi', 'Cairo'),

    # ===================== CAIRO – Heliopolis =====================
    'heliopolis': ('Heliopolis', 'Cairo'),
    'مصر الجديدة': ('Heliopolis', 'Cairo'),
    'merghany': ('Heliopolis', 'Cairo'),
    'المرغني': ('Heliopolis', 'Cairo'),

    # ===================== GIZA – Sheikh Zayed =====================
    'sheikh zayed': ('Sheikh Zayed', 'Giza'),
    'sheikhzayed': ('Sheikh Zayed', 'Giza'),
    'الشيخ زايد': ('Sheikh Zayed', 'Giza'),
    'youth housing': ('Sheikh Zayed', 'Giza'),
    'الاسكان الشبابي': ('Sheikh Zayed', 'Giza'),

    # ===================== GIZA – 6th of October =====================
    '6 october': ('6th Of October', 'Giza'),
    '6 أكتوبر': ('6th Of October', 'Giza'),
    '6thofoctober': ('6th Of October', 'Giza'),
    'october': ('6th Of October', 'Giza'),

    # ===================== GIZA – Dokki =====================
    'dokki': ('Dokki', 'Giza'),
    'الدقي': ('Dokki', 'Giza'),
    'tahrir': ('Dokki', 'Giza'),
    'التحرير': ('Dokki', 'Giza'),

    # ===================== GIZA – Mohandseen =====================
    'mohandseen': ('Mohandseen', 'Giza'),
    'المهندسين': ('Mohandseen', 'Giza'),
    'shehab': ('Mohandseen', 'Giza'),
    'lebanon': ('Mohandseen', 'Giza'),
    'لبنان': ('Mohandseen', 'Giza'),

    # ===================== ALEXANDRIA – Miami =====================
    'miami': ('Miami', 'Alexandria'),
    'ميامي': ('Miami', 'Alexandria'),
    'khaled ibn el walid': ('Miami', 'Alexandria'),
    'khaled ibnelwalid': ('Miami', 'Alexandria'),
    'khaledibnelwalid': ('Miami', 'Alexandria'),

    # ===================== ALEXANDRIA – Smouha =====================
    'smouha': ('Smouha', 'Alexandria'),
    'سموحة': ('Smouha', 'Alexandria'),
    'fawzy moaz': ('Smouha', 'Alexandria'),
    'fawzymoaz': ('Smouha', 'Alexandria'),
    'victor emanuel': ('Smouha', 'Alexandria'),
    'victoremanuel': ('Smouha', 'Alexandria'),

    # ===================== ALEXANDRIA – Lauran (+ generic "Alex" fallback) =====================
    'lauran': ('Lauran', 'Alexandria'),
    'لوران': ('Lauran', 'Alexandria'),
    'abdel salam aref': ('Lauran', 'Alexandria'),
    'abdelsalamaref': ('Lauran', 'Alexandria'),
    'alex': ('Lauran', 'Alexandria'),
    'alexandria': ('Lauran', 'Alexandria'),
    'الاسكندرية': ('Lauran', 'Alexandria'),
    'الإسكندرية': ('Lauran', 'Alexandria'),

    # ===================== DAKAHLIA – Mansoura =====================
    'mansoura': ('Mansoura', 'Dakahlia'),
    'المنصورة': ('Mansoura', 'Dakahlia'),
    'portsaid': ('Mansoura', 'Dakahlia'),   # street/area inside Mansoura, NOT Port Said governorate
    'بورسعيد': ('Mansoura', 'Dakahlia'),

    # ===================== ASSIUT – Assiut East =====================
    'assiut east': ('Assiut East', 'Assiut'),
    'assiuteast': ('Assiut East', 'Assiut'),
    'helaly': ('Assiut East', 'Assiut'),
    'الهلالي': ('Assiut East', 'Assiut'),

    # ===================== SUEZ – Arbeen =====================
    'arbeen': ('Arbeen', 'Suez'),
    'العربين': ('Arbeen', 'Suez'),
    'el geish': ('Arbeen', 'Suez'),   # also occurs under Assiut East, but 'assiut east' key above wins for those rows
    'elgeish': ('Arbeen', 'Suez'),

    # ===================== GENERIC GOVERNORATE FALLBACKS (lowest priority — keep last) =====================
    'giza': ('Giza', 'Giza'),
    'الجيزة': ('Giza', 'Giza'),
    'الجيزه': ('Giza', 'Giza'),
    'cairo': ('Cairo', 'Cairo'),
    'القاهرة': ('Cairo', 'Cairo'),
    'القاهره': ('Cairo', 'Cairo'),
    'assiut': ('Assiut', 'Assiut'),
    'اسيوط': ('Assiut', 'Assiut'),
    'أسيوط': ('Assiut', 'Assiut'),
    'suez': ('Suez', 'Suez'),
    'السويس': ('Suez', 'Suez'),
}


In [72]:
# Apply Mapping Function
# =======================================================
# Purpose: Scans the raw address for keywords from 'address_map'.
# Output:  Returns [City, Governorate] as a Pandas Series to create new columns.
# Fallback: Returns 'Unknown' if no keyword is found.

def map_address(addr):
    addr = str(addr).lower()
    for key, (city, gov) in address_map.items():
        if key in addr:
            return pd.Series([city, gov])
    return pd.Series(['Unknown', 'Unknown'])

In [73]:
# Execute Mapping & Create Columns
# =======================================================
# Purpose: Applies the mapping function to the entire 'address' column.
# Action:  Unpacks the results into two new clean columns: 'city' and 'governorate'.

df[['city', 'governorate']] = df['address'].apply(map_address)

In [74]:
df[['address', 'city', 'governorate']].head(10)

,address,city,governorate
0,"الجيزة, Sheikhzayed, Youth Housing, عمارة 12",Sheikh Zayed,Giza
1,"الدقهلية, Mansoura, Portsaid, عمارة 64",Mansoura,Dakahlia
2,"Assiut, Assiut East, El Geish, عمارة 23",Assiut East,Assiut
3,"Assiut, Assiut East, Helaly, Building 80",Assiut East,Assiut
4,"اسيوط, Assiuteast, El Geish, B60",Assiut East,Assiut
5,"الجيزة, Dokki, Tahrir, عمارة 6",Dokki,Giza
6,"السويس, Arbeen, El Geish, عمارة 24",Arbeen,Suez
7,"Suez, Arbeen, Elgeish, Building 53",Arbeen,Suez
8,"Alex, Miami, Khaledibnelwalid, Building 20",Miami,Alexandria
9,"السويس, Arbeen, Elgeish, Building 97",Arbeen,Suez


In [75]:
# Quality Assurance (QA) & Error Handling
# Purpose: Identifies any addresses that failed the mapping process.

df[df['city'] == 'Unknown']['address'].value_counts().head(10)


Series([], Name: count, dtype: int64)

In [76]:
# Export the fully cleaned and engineered customers dimension table to a new CSV file
df.to_csv("customers_cleaned.csv", index=False)


## Phase 5: Data Integration 

This final phase focuses on merging the cleaned Fact and Dimension tables into a single, flat denormalized dataset (OBT) ready for analysis.

**Key Objectives:**
* **Data Integration:** Merging the core `sales` table with `customers` and `projects` dimensions using their primary keys.
* **Denormalization:** Flattening the data into one comprehensive CSV to simplify querying and dashboard reporting.
* **Data Integrity:** Enforcing `Left Joins` across all merges to guarantee zero loss of core sales transactions.

In [77]:
import pandas as pd

sales_df = pd.read_csv("sales_cleaned.csv")
customers_df = pd.read_csv("customers_cleaned.csv")

# Merge Sales (Fact) with Customers (Dim) using Left Join to keep all transactions
sales_customers_df = pd.merge(
    sales_df,
    customers_df,
    on="customer_id",
    how="left"
)


In [78]:
sales_customers_df.head()


,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,...,customer_type,lead_source,budget_range,created_at,sales_agent,segment,created_year,created_month,city,governorate
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3Br,6691492,100000,6591492,8Y,938086.13,...,Individual,Referral,2-3M,2023-11-27,Ali,Vip,2023,11,Assiut East,Assiut
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,...,Company,Event,1-2M,2023-04-29,Ahmed,Normal,2023,4,Assiut East,Assiut
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,...,Individual,Whatsapp,3-5M,2025-07-27,Mohamed,New,2025,7,New Cairo,Cairo
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2Br,2363226,100000,2263226,Cash,554752.04,...,Company,Whatsapp,3-5M,2023-08-17,Ali,New,2023,8,Heliopolis,Cairo
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3Br,5212700,0,5212700,5Y,1536354.55,...,Company,Event,3-5M,2024-09-30,Ali,Vip,2024,9,New Cairo,Cairo


In [79]:
# Save the merged dataset to a new CSV (index=False prevents exporting the row numbers as an extra column)
sales_customers_df.to_csv("sales_with_customers.csv", index=False)


In [80]:
import pandas as pd

sales_customers_df = pd.read_csv("sales_with_customers.csv")
projects_df = pd.read_csv("projects_cleaned.csv")

# Merge the combined Sales & Customers data with the Projects Dimension table
final_df = pd.merge(
    sales_customers_df,
    projects_df,
    on="project_id",
    how="left"
)

In [81]:
final_df.shape

(1080000, 44)

In [82]:
# Save the final, fully merged Master Table to a new CSV (without exporting the row index)
final_df.to_csv("sales_full_dataset.csv", index=False)